**Mounting the google drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Unzipping the file in the worksheet**

In [ ]:
import zipfile, os

# This is relative to my device
with zipfile.ZipFile('/content/drive/MyDrive/picar/machine-learning-in-science-ii-2026.zip', 'r') as z:
    z.extractall('/content/')

# Check what came out
print(os.listdir('/content/'))

['.config', 'sample_submission.csv', 'drive', 'test_data', 'train.csv', 'training_data', 'sample_data']


**Imports**

In [ ]:
import os, random
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split

**Configuration**

In [ ]:
SEED         = 42
IMG_H        = 120
IMG_W        = 160
BATCH_SIZE   = 32
EPOCHS       = 40
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 8
TRAIN_CSV    = '/content/train.csv'
TRAIN_DIR = '/content/training_data/training_data'
TEST_DIR  = '/content/test_data/test_data'
OUTPUT_CSV   = '/content/submission.csv'
MODEL_PATH   = '/content/best_model.pt'
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f'Using device: {DEVICE}')

Using device: cuda


**Dataset class**

In [ ]:
class CarDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.img_dir   = Path(img_dir)
        self.transform = transform
        self.is_test   = is_test

        # Filter out any corrupted images upfront
        valid_rows = []
        for _, row in df.iterrows():
            img_path = self.img_dir / f"{int(row['image_id'])}.png"
            try:
                with Image.open(img_path) as img:
                    img.verify()  # check it's a valid image
                valid_rows.append(row)
            except Exception:
                print(f"Skipping corrupted image: {img_path}")

        self.df = pd.DataFrame(valid_rows).reset_index(drop=True)
        print(f"Dataset ready: {len(self.df)} valid images")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = self.img_dir / f"{int(row['image_id'])}.png"
        img      = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        if self.is_test:
            return img, int(row['image_id'])
        label = torch.tensor([float(row['angle']), float(row['speed'])], dtype=torch.float32) # label is a single tensor with both values
        return img, label

**Augmentation**

In [ ]:
def get_transforms(augment):
    base = [
        transforms.Resize((IMG_H, IMG_W)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ]
    if augment:
        aug = [
            # This is only applied during training - This is to artificially expand the diversity of the training
            # data by showing the model slightly modified versions of each image.

            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),  # varies brightness, contrast, saturation
            transforms.RandomAffine(degrees=5, translate=(0.05, 0.05)),  #applies small random rotations
        ]
        return transforms.Compose(aug + base)
    return transforms.Compose(base)

**Model**

In [ ]:
# Instead of building the CNN from scratch, we start with MobileNetV3-Small — it is  a model that was already trained on "ImageNet"
# We then replace its final classification layer with our own regression head
# The pretrained backbone already understands visual features, so we just need to teach the new head to map those features to steering angles and speeds

class PiCarNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        backbone = models.mobilenet_v3_small(
            weights=models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        )
        in_features = backbone.classifier[0].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Dropout(p=0.3), # Two drop out layers : 30% - is aggressive enough to prevent the large backbone features from overfitting
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(p=0.2), #20% - adds a lighter touch of regularisation before the final output
            nn.Linear(128, 2), # Outputs two numbers simultaneously
            nn.Sigmoid(), # both squashed to [0, 1] - index 0 is angle and index 1 is speed
        )

    def forward(self, x):
        return self.head(self.backbone(x))

**Training helpers**

In [ ]:
def mse_loss(pred, target):
    return nn.functional.mse_loss(pred, target) # Averages the error across both outputs

def train_one_epoch(model, loader, optimiser, scaler):
    model.train()
    total_loss = 0.0
    for imgs, labels in tqdm(loader, desc='  train', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimiser.zero_grad()
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
            preds = model(imgs)
            loss  = mse_loss(preds, labels)
        scaler.scale(loss).backward()
        scaler.step(optimiser)
        scaler.update()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    for imgs, labels in tqdm(loader, desc='  valid', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        preds = model(imgs)
        total_loss += mse_loss(preds, labels).item() * imgs.size(0)
    return total_loss / len(loader.dataset)

**Data Loading**

In [ ]:
df = pd.read_csv(TRAIN_CSV)
print(f'Training labels loaded: {len(df)} rows')
print(df.head())

train_df, val_df = train_test_split(df, test_size=0.15, random_state=SEED, shuffle=True)
print(f'Train: {len(train_df)}   Val: {len(val_df)}')

train_ds = CarDataset(train_df, TRAIN_DIR, transform=get_transforms(augment=True))
val_ds   = CarDataset(val_df,   TRAIN_DIR, transform=get_transforms(augment=False))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print('Data loaders ready!')

Training labels loaded: 14419 rows
   image_id   angle  speed
0         0  0.0625    1.0
1         1  0.5000    1.0
2         2  0.6250    1.0
3         3  0.4375    1.0
4         4  0.5625    1.0
Train: 12256   Val: 2163
Skipping corrupted image: /content/training_data/training_data/2968.png
Skipping corrupted image: /content/training_data/training_data/2174.png
Skipping corrupted image: /content/training_data/training_data/4664.png
Skipping corrupted image: /content/training_data/training_data/9657.png
Skipping corrupted image: /content/training_data/training_data/9630.png
Skipping corrupted image: /content/training_data/training_data/5794.png
Skipping corrupted image: /content/training_data/training_data/1612.png
Skipping corrupted image: /content/training_data/training_data/58.png
Skipping corrupted image: /content/training_data/training_data/10146.png
Skipping corrupted image: /content/training_data/training_data/3732.png
Skipping corrupted image: /content/training_data/training_d

**Building model**

In [ ]:
# Using the AdamW - It is an adaptive gradient descent algorithm
# It maintains a per-parameter learning rate that adapts based on the history of gradients
# The W means it applies weight decay

model     = PiCarNet(pretrained=True).to(DEVICE)
optimiser = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Instead of keeping the learning rate fixed at 0.001 throughout all 40 epochs, this smoothly reduces it following a cosine curve
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=EPOCHS)

# Mixed Precision Training - It switches the forward pass to 16-bit, which is twice as fast and uses half the memory,
# while keeping the weight updates in 32-bit for numerical stability
scaler    = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))

print('Model built')

NameError: name 'PiCarNet' is not defined

**Training loop**

In [ ]:
best_val   = float('inf')
no_improve = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimiser, scaler)
    val_loss   = evaluate(model, val_loader)
    scheduler.step()

    print(f'Epoch {epoch:3d}/{EPOCHS}  train_MSE={train_loss:.6f}  val_MSE={val_loss:.6f}  lr={scheduler.get_last_lr()[0]:.2e}')

    # After every epoch, if the validation MSE is better than any previous epoch, the model weights are saved to disk
    # At the end, the best saved weights are reloaded, in this case was: epoch 37
    if val_loss < best_val:
        best_val   = val_loss
        no_improve = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'  checkmark Saved best model (val_MSE={best_val:.6f})')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'  Early stopping after {PATIENCE} epochs without improvement.')
            break

print(f'Best validation MSE: {best_val:.6f}')

  train:   0%|          | 0/383 [00:00<?, ?it/s]/tmp/ipykernel_4953/303298138.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):


Epoch   1/40  train_MSE=0.030964  val_MSE=0.018001  lr=9.98e-04
  checkmark Saved best model (val_MSE=0.018001)


Epoch   2/40  train_MSE=0.018539  val_MSE=0.020354  lr=9.94e-04


Epoch   3/40  train_MSE=0.017512  val_MSE=0.016068  lr=9.86e-04
  checkmark Saved best model (val_MSE=0.016068)


Epoch   4/40  train_MSE=0.016625  val_MSE=0.019371  lr=9.76e-04


...

Epoch  36/40  train_MSE=0.006567  val_MSE=0.009650  lr=2.45e-05


Epoch  37/40  train_MSE=0.006568  val_MSE=0.009557  lr=1.38e-05
  checkmark Saved best model (val_MSE=0.009557)


Epoch  38/40  train_MSE=0.006708  val_MSE=0.009632  lr=6.16e-06


Epoch  39/40  train_MSE=0.006446  val_MSE=0.009632  lr=1.54e-06


Epoch  40/40  train_MSE=0.006519  val_MSE=0.009616  lr=0.00e+00
Best validation MSE: 0.009557


**Generate submission**

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

test_ids    = sorted([int(f.stem) for f in Path(TEST_DIR).glob('*.png')])
test_df     = pd.DataFrame({'image_id': test_ids})
test_ds     = CarDataset(test_df, TEST_DIR, transform=get_transforms(augment=False), is_test=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

results = []
with torch.no_grad():
    for imgs, ids in tqdm(test_loader, desc='Inference'):
        imgs  = imgs.to(DEVICE)
        preds = model(imgs).cpu().numpy()
        for img_id, pred in zip(ids.numpy(), preds):
            results.append({
                'image_id': int(img_id),
                'angle':    float(np.clip(pred[0], 0, 1)),
                'speed':    float(np.clip(pred[1], 0, 1)),
            })

sub = pd.DataFrame(results).sort_values('image_id')
sub.to_csv(OUTPUT_CSV, index=False)
print(f'Submission saved to {OUTPUT_CSV} ({len(sub)} rows)')
print(sub.head(10))

Dataset ready: 2000 valid images


Inference: 100%|██████████| 63/63 [00:07<00:00,  8.32it/s]

Submission saved to /content/submission.csv (2000 rows)
   image_id     angle     speed
0         0  0.511250  0.033463
1         1  0.681657  0.999990
2         2  0.713538  1.000000
3         3  0.761029  0.000186
4         4  0.508574  1.000000
5         5  0.473396  1.000000
6         6  0.592721  0.999572
7         7  0.549670  0.771431
8         8  0.503658  1.000000
9         9  0.499649  0.003941


**Submission file**

In [ ]:
from google.colab import files
files.download('/content/submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>